# Long neural training with strength monitoring

**Default: preflight only.** Set `RUN_MODE="trial"` for 400 additional games (evaluation after 200 and 400), `RUN_MODE="medium"` for 2,000, or `RUN_MODE="full"` for 10,000; also set `START_TRAINING=True` to launch. This notebook has not been executed by Codex. It continues the saved model, optimizer, schedule and opponent archive; it does not start from random weights. Set `START_TRAINING = False` if you only want setup/preflight.

PPO updates every **8 games**. Durable model/opponent snapshots save roughly every **128 games**, plus the final batch. Fixed-opponent evaluation runs before learning, roughly every **512 training games**, and at the end. Evaluation uses 64 games per opponent on a separate monitoring seed set. All old runs are preserved. Estimated model/checkpoint storage for this preset is about **6 GB**, printed precisely by preflight; each launch creates a new folder. Interrupting loses progress since the last durable checkpoint (up to 128 games).

The dashboard shows completed/capped games, elapsed time, rough remaining time, PPO updates, fixed-opponent win/draw/loss results, approximate 95% win-rate intervals, a score curve and checkpoint paths. Eight-game batches are repeated continuously until the full budget is consumed; they are not isolated restarts.

The opponents are a transparent hand-written heuristic, the original untrained neural model and the saved starting checkpoint. They remain fixed across evaluation reports and exact resumes. Beating these baselines does **not** establish expert human strength. Reused monitoring seeds are validation, not an unbiased final test. Simulator approximations still apply.


In [ ]:
from pathlib import Path
import json, subprocess, sys
from IPython.display import HTML, display
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'tools/train_neural.py').exists()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook inside the complete death-knight-lab checkout.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from hearthstone_sim.neural.dashboard import Dashboard

RUN_MODE = "preflight"  # preflight, trial, medium, full
START_TRAINING = False  # Run All is an explicit request to launch the long run.
TOTAL_TRAINING_GAMES = 400 if RUN_MODE == "trial" else 2_000 if RUN_MODE == "medium" else 10_000  # ADDITIONAL games, not including evaluation.
GAMES_PER_ITERATION = 8
MAX_ACTIONS = 1000
SEED = 71
DEVICE = 'cpu'
THREADS = 2  # PyTorch threads only; games remain sequential.
FAILURE_FRACTION = 0.02
FAILURE_MIN_ATTEMPTS = 100
FAILURE_REPEATS = 5
RETRY_CAP = 20
TRACE_FIRST = 6
TRACE_EVERY = 16
assert RUN_MODE in ("preflight", "trial", "medium", "full")
EVALUATE_AT_START = True  # Resumed weights at local game 0, before any updates.
CHECKPOINT_EVERY_ITERATIONS = 16  # 128 games; smaller intervals use more disk.
EVALUATION_EVERY_ITERATIONS = 64  # Roughly every 512 training games.
EVALUATION_GAMES_PER_OPPONENT = 64
EVALUATION_SEED = 1_000_000_000
RESUME_CHECKPOINT = None  # Set to your own compatible checkpoint to continue a saved run.
ALLOW_CODE_UPGRADE = False  # Keep strict compatibility checks for normal resumes.
assert TOTAL_TRAINING_GAMES > 0 and TOTAL_TRAINING_GAMES % GAMES_PER_ITERATION == 0
ITERATIONS = TOTAL_TRAINING_GAMES // GAMES_PER_ITERATION
command = [sys.executable, str(ROOT / 'tools/train_neural.py'),
    '--iterations', str(ITERATIONS), '--games-per-iteration', str(GAMES_PER_ITERATION),
    '--max-actions', str(MAX_ACTIONS), '--seed', str(SEED), '--device', DEVICE,
    '--threads', str(THREADS), '--checkpoint-every-iterations', str(CHECKPOINT_EVERY_ITERATIONS),
    '--evaluation-every-iterations', str(EVALUATION_EVERY_ITERATIONS),
    '--evaluation-games-per-opponent', str(EVALUATION_GAMES_PER_OPPONENT),
    '--evaluation-seed', str(EVALUATION_SEED)]
if RESUME_CHECKPOINT is not None:
    command += ['--resume', str(RESUME_CHECKPOINT)]
if ALLOW_CODE_UPGRADE:
    command += ['--allow-code-upgrade']
command += ['--failure-fraction', str(FAILURE_FRACTION), '--failure-min-attempts', str(FAILURE_MIN_ATTEMPTS), '--failure-repeats', str(FAILURE_REPEATS), '--retry-cap', str(RETRY_CAP), '--trace-first', str(TRACE_FIRST), '--trace-every', str(TRACE_EVERY)]
command += ['--evaluate-at-start' if EVALUATE_AT_START else '--no-evaluate-at-start']
if RUN_MODE == 'trial':
    command += ['--evaluation-at', '200', '400']
elif RUN_MODE == 'medium':
    # Local batch iterations 64, 128, 192, 250, plus baseline at 0.
    command += ['--evaluation-at', '512', '1024', '1536', '2000']
print(f'Ready to check: {TOTAL_TRAINING_GAMES:,} additional training games; {ITERATIONS:,} PPO batches.')
print('No training or evaluation has started in this cell.')



In [ ]:
# Read-only validation: no games, no evaluation and no weight updates.
preflight = subprocess.run(command + ['--preflight'], cwd=ROOT, text=True,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(preflight.stdout)
PREFLIGHT_OK = preflight.returncode == 0
if not PREFLIGHT_OK:
    raise RuntimeError('Preflight failed. The training cell will refuse to launch.')


In [ ]:
if not START_TRAINING or RUN_MODE == "preflight":
    print('Preflight only. Set RUN_MODE to trial or full and START_TRAINING to True to launch.')
else:
    dashboard = Dashboard(TOTAL_TRAINING_GAMES)
    screen = display(HTML(dashboard.html()), display_id=True)
    LAST_CHECKPOINT = None
    RUN_FOLDER = None
    succeeded = False
    with subprocess.Popen(command + ['--train'], cwd=ROOT, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True) as process:
        try:
            for line in process.stdout:
                try:
                    event = json.loads(line)
                except ValueError:
                    print(line.rstrip())
                    continue
                dashboard.consume(event)
                screen.update(HTML(dashboard.html()))
                if event['event'] == 'started':
                    RUN_FOLDER = event['folder']
                    print('Run folder:', RUN_FOLDER)
                elif event['event'] == 'iteration_saved' and event.get('checkpoint'):
                    LAST_CHECKPOINT = event['checkpoint']['path']
                    print('Resume checkpoint:', LAST_CHECKPOINT)
                elif event['event'] == 'evaluation_finished':
                    print('Detailed matchup evaluation:', event['report'])
                elif event['event'] == 'complete':
                    succeeded = True
                    LAST_CHECKPOINT = event['checkpoint']['path']
            returncode = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            dashboard.status = 'Interrupted — resume from the latest durable checkpoint'
            screen.update(HTML(dashboard.html()))
            raise
    if returncode or not succeeded:
        dashboard.status = 'Stopped with an error — see output above; earlier checkpoints are preserved'
        screen.update(HTML(dashboard.html()))
        raise RuntimeError('Run did not finish. Do not delete the run folder; use the latest printed checkpoint.')
    print('Completed. Set RESUME_CHECKPOINT to this path for your next launch:', LAST_CHECKPOINT)


## Resume and interpretation

To continue an interrupted or completed long run, use its latest printed `training-*.pt` path as `RESUME_CHECKPOINT`, keep the other settings unchanged, and set `TOTAL_TRAINING_GAMES` to your next additional budget. Keep the **entire run folder**, including opponent snapshots and `.json` hash receipts. Exact resumes restore Adam, sampling RNGs and scheduling offsets.

The first continuation explicitly upgrades the old training harness. It preserves weights, Adam, RNGs, eight already completed games and the archive. It permits **only** that known old code version with the unchanged engine, model, observation encoder, PPO, decks and dependency versions. It is labeled a training-code upgrade, not bitwise exact replay of the old harness. Once a new checkpoint is saved, normal exact resume applies. No hash or old checkpoint file is rewritten.

The first evaluation establishes the starting score; it is not improvement evidence. Look for sustained gains with uncertainty considered across multiple reports and deck matchups. Cap-heavy evaluations are not reliable evidence, even if the completed-game win rate rises. The score curve counts a win as 1, a draw as ½ and a loss as 0. The table's Wilson intervals describe win rate, counting draws as non-wins. These are approximate for a fixed matchup panel. Full expert-player claims require stronger independent opponents and fresh final-test seeds.

Monitoring games never enter PPO and do not advance its RNG streams. The simulator still has documented approximations; a policy can exploit them. No game count guarantees strong play. The preset has a finite explicit budget and does not launch another run on completion.
